# 🧠 EEG Neuroprosthetic Control — Grasp-and-Lift Detection

**Objectif :** Décoder l'intention motrice à partir de signaux EEG bruts pour le contrôle neuroprosthétique  
**Dataset :** [Kaggle — Grasp-and-Lift EEG Detection](https://www.kaggle.com/competitions/grasp-and-lift-eeg-detection)  
**Auteur :** [Votre nom]  
**Date :** 2025

---

## Contexte clinique

Le contrôle d'une main prothétique nécessite une détection fiable et **en temps quasi-réel** des intentions motrices à partir de signaux EEG. Ce notebook explore un pipeline complet allant du signal brut jusqu'à la classification des événements moteurs, en intégrant une dimension critique souvent absente des approches purement académiques : **la viabilité temps-réel** des modèles.

### Les 6 événements moteurs à détecter
| # | Événement | Pertinence neuroprosthétique |
|---|-----------|------------------------------|
| 1 | HandStart | Initiation du mouvement |
| 2 | FirstDigitTouch | Contact avec l'objet |
| 3 | BothStartLoadPhase | Début de la charge |
| 4 | LiftOff | Décollage de l'objet |
| 5 | Replace | Repose de l'objet |
| 6 | BothReleased | Relâchement final |

> **Note importante :** Les labels sont actifs si l'événement se produit dans une fenêtre de ±150ms autour du timestamp courant. Cela définit la **contrainte temporelle réelle** du système.

---

## Structure du notebook
1. [Setup & Imports](#1-setup)
2. [Exploration du dataset](#2-exploration)
3. [Preprocessing du signal EEG](#3-preprocessing)
4. [Feature Extraction](#4-features)
5. [Classification — Approche ML](#5-ml)
6. [Benchmark Temps-Réel](#6-benchmark)
7. [Discussion & Applicabilité Clinique](#7-discussion)

---
## 1. Setup & Imports <a id='1-setup'></a>

In [ ]:
# ── Installation des dépendances (Kaggle environment) ──────────────────────────
# !pip install mne scipy scikit-learn pandas numpy matplotlib seaborn tqdm

# ── Imports standard ───────────────────────────────────────────────────────────
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm
warnings.filterwarnings('ignore')

# ── Signal processing ──────────────────────────────────────────────────────────
from scipy import signal
from scipy.signal import butter, filtfilt, iirnotch

# ── Machine Learning ───────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    RocCurveDisplay
)
from sklearn.multioutput import MultiOutputClassifier

# ── Config visuelle ────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = plt.cm.tab10.colors
sns.set_palette('tab10')

# ── Constantes dataset ─────────────────────────────────────────────────────────
FS          = 500          # Fréquence d'échantillonnage (Hz)
N_CHANNELS  = 32           # Canaux EEG
N_SUBJECTS  = 12           # Sujets
N_EVENTS    = 6            # Classes d'événements
TRAIN_SERIES = [1,2,3,4,5,6,7,8]  # Séries d'entraînement
TEST_SERIES  = [9, 10]             # Séries de test
EVENT_NAMES  = ['HandStart','FirstDigitTouch','BothStartLoadPhase',
                'LiftOff','Replace','BothReleased']
EVENT_WINDOW_MS = 150      # Fenêtre de labellisation (±150ms)

# ── Chemins ────────────────────────────────────────────────────────────────────
DATA_DIR = '/kaggle/input/grasp-and-lift-eeg-detection'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR  = os.path.join(DATA_DIR, 'test')

print(f"Setup OK — Fs={FS}Hz | {N_CHANNELS} canaux | {N_SUBJECTS} sujets | {N_EVENTS} événements")
print(f"Fenêtre de labellisation : ±{EVENT_WINDOW_MS}ms → contrainte temps-réel principale")

---
## 2. Exploration du dataset <a id='2-exploration'></a>

In [ ]:
# ── Listage des fichiers disponibles ───────────────────────────────────────────
train_files = sorted(os.listdir(TRAIN_DIR))
data_files  = [f for f in train_files if 'data'   in f]
event_files = [f for f in train_files if 'events' in f]

print(f"Fichiers data   : {len(data_files)}")
print(f"Fichiers events : {len(event_files)}")
print(f"\nExemple : {data_files[0]}")

In [ ]:
# ── Fonction de chargement ─────────────────────────────────────────────────────
def load_subject_series(subject_id, series_id, data_dir=TRAIN_DIR):
    """
    Charge les données EEG et les labels pour un sujet/série donné.
    
    Returns:
        X (np.ndarray) : signal EEG [n_samples x n_channels]
        y (np.ndarray) : labels [n_samples x 6] — multilabel binaire
        channels (list): noms des 32 canaux
    """
    data_path  = os.path.join(data_dir,  f'subj{subject_id}_series{series_id}_data.csv')
    event_path = os.path.join(data_dir,  f'subj{subject_id}_series{series_id}_events.csv')
    
    df_data   = pd.read_csv(data_path)
    df_events = pd.read_csv(event_path)
    
    channels = [c for c in df_data.columns if c != 'id']
    X = df_data[channels].values.astype(np.float32)
    y = df_events[EVENT_NAMES].values.astype(np.int8)
    
    return X, y, channels


# ── Chargement sujet 1, série 1 pour exploration ───────────────────────────────
X_demo, y_demo, channels = load_subject_series(subject_id=1, series_id=1)

duration_s = X_demo.shape[0] / FS
print(f"Dimensions EEG   : {X_demo.shape}  ({X_demo.shape[0]} samples × {X_demo.shape[1]} canaux)")
print(f"Dimensions labels: {y_demo.shape}")
print(f"Durée de la série: {duration_s:.1f}s ({duration_s/60:.1f} min)")
print(f"\nCanaux disponibles:")
print(channels)

In [ ]:
# ── Distribution des événements ────────────────────────────────────────────────
event_counts = y_demo.sum(axis=0)
event_rates  = event_counts / y_demo.shape[0] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Barplot
axes[0].bar(EVENT_NAMES, event_counts, color=COLORS[:6], alpha=0.85, edgecolor='white')
axes[0].set_title('Nombre de frames actives par événement (Sujet 1, Série 1)', fontweight='bold')
axes[0].set_ylabel('Frames actives')
axes[0].tick_params(axis='x', rotation=30)

# Taux d'activation
axes[1].bar(EVENT_NAMES, event_rates, color=COLORS[:6], alpha=0.85, edgecolor='white')
axes[1].set_title('Taux d\'activation (%) par événement', fontweight='bold')
axes[1].set_ylabel('% frames actives')
axes[1].tick_params(axis='x', rotation=30)
axes[1].axhline(50, ls='--', color='gray', alpha=0.5, label='50%')

plt.tight_layout()
plt.savefig('event_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n→ Déséquilibre de classes attendu : les événements sont rares (tâche naturelle)")

In [ ]:
# ── Visualisation des signaux bruts ────────────────────────────────────────────
# Canaux moteurs d'intérêt (standard 10-20)
MOTOR_CHANNELS = ['C3', 'Cz', 'C4', 'CP1', 'CP2']  
motor_idx = [channels.index(c) for c in MOTOR_CHANNELS if c in channels]

t = np.arange(X_demo.shape[0]) / FS  # axe temporel en secondes

# Affichage sur 30 secondes
t_start, t_end = 0, 30
mask = (t >= t_start) & (t <= t_end)

fig, axes = plt.subplots(len(motor_idx) + 1, 1, figsize=(16, 10), sharex=True)
fig.suptitle(f'Signaux EEG bruts — Canaux moteurs (Sujet 1, Série 1, {t_start}-{t_end}s)',
             fontweight='bold', fontsize=13)

# Signaux EEG
for i, idx in enumerate(motor_idx):
    axes[i].plot(t[mask], X_demo[mask, idx], color=COLORS[i], lw=0.6, alpha=0.9)
    axes[i].set_ylabel(MOTOR_CHANNELS[i], fontsize=10, rotation=0, labelpad=35)
    axes[i].set_ylim(np.percentile(X_demo[:, idx], 1), np.percentile(X_demo[:, idx], 99))

# Étiquettes événements
ax_events = axes[-1]
for j, ev in enumerate(EVENT_NAMES):
    ev_mask = y_demo[mask, j].astype(bool)
    ax_events.fill_between(t[mask], j, j + 0.8,
                           where=ev_mask, color=COLORS[j], alpha=0.7, label=ev)
ax_events.set_yticks(np.arange(len(EVENT_NAMES)) + 0.4)
ax_events.set_yticklabels(EVENT_NAMES, fontsize=8)
ax_events.set_xlabel('Temps (s)')
ax_events.set_ylabel('Événements', fontsize=10)

plt.tight_layout()
plt.savefig('raw_signals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Analyse statistique du dataset complet ─────────────────────────────────────
print("=" * 55)
print("AUDIT DU DATASET")
print("=" * 55)

total_frames = 0
event_totals = np.zeros(N_EVENTS, dtype=int)

for subj in range(1, N_SUBJECTS + 1):
    for serie in TRAIN_SERIES:
        try:
            X_tmp, y_tmp, _ = load_subject_series(subj, serie)
            total_frames  += X_tmp.shape[0]
            event_totals  += y_tmp.sum(axis=0)
        except FileNotFoundError:
            pass

print(f"\nTotal frames (train)  : {total_frames:,}")
print(f"Durée totale          : {total_frames / FS / 3600:.1f} heures")
print(f"\nDistribution des événements (train) :")
for ev, cnt in zip(EVENT_NAMES, event_totals):
    pct = cnt / total_frames * 100
    print(f"  {ev:<25} : {cnt:>8,} frames ({pct:.1f}%)")

---
## 3. Preprocessing du signal EEG <a id='3-preprocessing'></a>

### Stratégie de filtrage

Pour la détection d'événements moteurs (grip/lift), les bandes fréquentielles pertinentes sont :
- **Bande mu (8-12 Hz)** : désynchronisation contralaterale lors du mouvement (ERD)
- **Bande bêta (13-30 Hz)** : rebond post-mouvement (ERS)
- **Basses fréquences (<5 Hz)** : Bereitschaftspotential (potentiel de préparation motrice)

Le vainqueur de la compétition (Alexandre Barachant) a montré que les **basses fréquences** sont particulièrement discriminantes pour cette tâche.

In [ ]:
# ── Fonctions de filtrage ──────────────────────────────────────────────────────

def butter_bandpass(lowcut, highcut, fs=FS, order=5):
    """Filtre Butterworth passe-bande."""
    nyq = 0.5 * fs
    low, high = lowcut / nyq, highcut / nyq
    return butter(order, [low, high], btype='band')

def butter_lowpass(cutoff, fs=FS, order=5):
    """Filtre Butterworth passe-bas."""
    nyq  = 0.5 * fs
    return butter(order, cutoff / nyq, btype='low')

def apply_filter(X, b, a):
    """Application zero-phase (filtfilt) sur toutes les colonnes."""
    return filtfilt(b, a, X, axis=0)

def notch_filter(X, freq=50, fs=FS, Q=30):
    """Notch filter pour éliminer le bruit secteur (50 Hz)."""
    b, a = iirnotch(freq, Q, fs)
    return filtfilt(b, a, X, axis=0)


def preprocess_signal(X, strategy='bandpass_mu_beta'):
    """
    Pipeline de preprocessing modulaire.
    
    Strategies disponibles :
      - 'bandpass_mu_beta' : passe-bande 8-30 Hz (mu + bêta)
      - 'lowpass'          : passe-bas 30 Hz (approche large bande)
      - 'lowpass_lf'       : passe-bas 5 Hz (basses fréquences — approche vainqueur)
      - 'car'              : Common Average Reference uniquement
    """
    X_proc = X.copy()
    
    # 1. Notch 50 Hz
    X_proc = notch_filter(X_proc)
    
    # 2. Filtre fréquentiel selon stratégie
    if strategy == 'bandpass_mu_beta':
        b, a = butter_bandpass(8, 30)
        X_proc = apply_filter(X_proc, b, a)
    elif strategy == 'lowpass':
        b, a = butter_lowpass(30)
        X_proc = apply_filter(X_proc, b, a)
    elif strategy == 'lowpass_lf':
        b, a = butter_lowpass(5)
        X_proc = apply_filter(X_proc, b, a)
    
    # 3. Re-référencement CAR (Common Average Reference)
    X_proc = X_proc - X_proc.mean(axis=1, keepdims=True)
    
    # 4. Normalisation z-score par canal (par série)
    X_proc = (X_proc - X_proc.mean(axis=0)) / (X_proc.std(axis=0) + 1e-8)
    
    return X_proc


# ── Application sur les données de démo ───────────────────────────────────────
X_filt_mb = preprocess_signal(X_demo, strategy='bandpass_mu_beta')
X_filt_lf = preprocess_signal(X_demo, strategy='lowpass_lf')

print("Preprocessing appliqué avec succès")
print(f"Signal filtré (mu+bêta) : {X_filt_mb.shape} | mean={X_filt_mb.mean():.4f} | std={X_filt_mb.std():.4f}")
print(f"Signal filtré (LF <5Hz) : {X_filt_lf.shape} | mean={X_filt_lf.mean():.4f} | std={X_filt_lf.std():.4f}")

In [ ]:
# ── Comparaison signal brut vs filtré ──────────────────────────────────────────
ch_plot = 'C3'
ch_idx  = channels.index(ch_plot)
t_win   = (t >= 5) & (t <= 10)  # Fenêtre 5-10s

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
fig.suptitle(f'Effet du filtrage sur le canal {ch_plot}', fontweight='bold', fontsize=13)

axes[0].plot(t[t_win], X_demo[t_win, ch_idx], color='steelblue', lw=0.8)
axes[0].set_title('Signal brut')
axes[0].set_ylabel('µV')

axes[1].plot(t[t_win], X_filt_mb[t_win, ch_idx], color='darkorange', lw=0.8)
axes[1].set_title('Filtré passe-bande mu+bêta (8-30 Hz)')
axes[1].set_ylabel('z-score')

axes[2].plot(t[t_win], X_filt_lf[t_win, ch_idx], color='forestgreen', lw=0.8)
axes[2].set_title('Filtré basses fréquences (<5 Hz) — Bereitschaftspotential')
axes[2].set_ylabel('z-score')
axes[2].set_xlabel('Temps (s)')

plt.tight_layout()
plt.savefig('filtering_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Densité spectrale de puissance (PSD) ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PSD — Canal C3 (Sujet 1, Série 1)', fontweight='bold')

for ax, X_plot, label, color in [
    (axes[0], X_demo,      'Signal brut',           'steelblue'),
    (axes[1], X_filt_mb,   'Filtré (8-30 Hz)',       'darkorange'),
]:
    freqs, psd = signal.welch(X_plot[:, ch_idx], fs=FS, nperseg=FS*2)
    ax.semilogy(freqs, psd, color=color, lw=1.5)
    ax.set_xlim([0, 60])
    ax.set_title(label)
    ax.set_xlabel('Fréquence (Hz)')
    ax.set_ylabel('PSD (µV²/Hz)')
    # Annotation des bandes
    for band, (fmin, fmax), col in [
        ('δ', (1,4), 'purple'), ('θ', (4,8), 'blue'),
        ('μ', (8,12), 'green'), ('β', (13,30), 'red')
    ]:
        ax.axvspan(fmin, fmax, alpha=0.08, color=col, label=band)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('psd_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Feature Extraction <a id='4-features'></a>

### Approche par fenêtres glissantes

Pour chaque fenêtre temporelle, on extrait des features capturant :
- **L'énergie dans les bandes fréquentielles** (mu, bêta, basses fréquences)
- **Les matrices de covariance** (information spatiale inter-canaux — approche gagnante de la compétition)
- **Des statistiques temporelles** (variance, kurtosis)

> **Paramètre critique pour le temps-réel :** la taille de la fenêtre glissante.
> - Grande fenêtre (ex: 2000 samples = 4s) → bonne accuracy, latence incompatible
> - Petite fenêtre (ex: 125 samples = 250ms) → latence faible, accuracy dégradée
> - **On testera les deux** pour quantifier ce trade-off.

In [ ]:
# ── Extraction de features par fenêtre glissante ───────────────────────────────

def compute_band_power(x_win, fs=FS):
    """
    Calcule la puissance dans les bandes mu, bêta et basses fréquences.
    x_win : signal 1D (une fenêtre temporelle, un canal)
    """
    freqs, psd = signal.welch(x_win, fs=fs, nperseg=min(len(x_win), fs//2))
    
    def band_power(fmin, fmax):
        idx = (freqs >= fmin) & (freqs <= fmax)
        return np.trapz(psd[idx], freqs[idx])
    
    return [
        band_power(1, 4),    # delta
        band_power(4, 8),    # theta
        band_power(8, 12),   # mu
        band_power(13, 30),  # beta
        band_power(30, 50),  # gamma
    ]


def extract_features_window(X_win):
    """
    Extrait les features d'une fenêtre EEG [n_samples x n_channels].
    
    Features :
      - Band power (5 bandes × 32 canaux) = 160 features
      - Variance par canal = 32 features
      - Éléments upper-triangulaire de la matrice de covariance = 528 features
    Total ≈ 720 features
    """
    features = []
    
    # 1. Band power (5 bandes × N_channels)
    for ch_idx in range(X_win.shape[1]):
        bp = compute_band_power(X_win[:, ch_idx])
        features.extend(bp)
    
    # 2. Variance par canal
    features.extend(np.var(X_win, axis=0).tolist())
    
    # 3. Matrice de covariance (upper triangle)
    cov = np.cov(X_win.T)
    triu_idx = np.triu_indices(cov.shape[0])
    features.extend(cov[triu_idx].tolist())
    
    return np.array(features, dtype=np.float32)


def sliding_window_features(X, y, window_size, step_size):
    """
    Découpe X en fenêtres glissantes et extrait les features.
    Label = label au centre de la fenêtre.
    
    Args:
        X           : signal EEG [n_samples x n_channels]
        y           : labels [n_samples x 6]
        window_size : taille de la fenêtre (samples)
        step_size   : pas du glissement (samples)
    
    Returns:
        X_feat [n_windows x n_features]
        y_win  [n_windows x 6]
    """
    X_feat, y_win = [], []
    half_win = window_size // 2
    
    for start in range(0, X.shape[0] - window_size, step_size):
        end    = start + window_size
        center = start + half_win
        
        feat  = extract_features_window(X[start:end])
        label = y[center]
        
        X_feat.append(feat)
        y_win.append(label)
    
    return np.array(X_feat), np.array(y_win)


print("Fonctions de feature extraction définies.")
print("\nConfiguration des fenêtres à tester :")
WINDOW_CONFIGS = {
    'long_4s'   : {'window': 4*FS,   'step': FS//2,  'label': 'Fenêtre 4s   (haute accuracy)'},
    'medium_1s' : {'window': 1*FS,   'step': FS//4,  'label': 'Fenêtre 1s   (compromis)'},
    'short_250ms': {'window': FS//4, 'step': FS//10, 'label': 'Fenêtre 250ms (temps-réel)'},
}
for k, cfg in WINDOW_CONFIGS.items():
    latency_ms = cfg['window'] / FS * 1000
    print(f"  {cfg['label']} → latence = {latency_ms:.0f}ms")

In [ ]:
# ── Extraction sur un sujet (démonstration) ────────────────────────────────────
# Note : Pour l'entraînement complet, étendre à tous les sujets

print("Extraction des features (Sujet 1, Séries 1-6) — patience...")

features_by_window = {}

for config_name, cfg in WINDOW_CONFIGS.items():
    X_all_feat, y_all = [], []
    
    for serie in [1, 2, 3, 4, 5, 6]:  # Train series pour sujet 1
        try:
            X_s, y_s, _ = load_subject_series(1, serie)
            X_s = preprocess_signal(X_s, strategy='lowpass_lf')  # Stratégie LF
            X_f, y_f = sliding_window_features(X_s, y_s,
                                               window_size=cfg['window'],
                                               step_size=cfg['step'])
            X_all_feat.append(X_f)
            y_all.append(y_f)
        except Exception as e:
            print(f"  Série {serie} — erreur: {e}")
    
    X_concat = np.vstack(X_all_feat)
    y_concat = np.vstack(y_all)
    features_by_window[config_name] = (X_concat, y_concat)
    
    print(f"  {cfg['label']}: {X_concat.shape[0]} fenêtres × {X_concat.shape[1]} features")

---
## 5. Classification — Approche ML <a id='5-ml'></a>

On évalue plusieurs classifieurs sur chaque configuration de fenêtre.  
**Métrique principale :** AUC ROC (métrique officielle de la compétition Kaggle).  
**Validation :** Leave-one-series-out (évite la fuite de données temporelles).

In [ ]:
# ── Définition des modèles ─────────────────────────────────────────────────────
MODELS = {
    'LDA' : Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LinearDiscriminantAnalysis())
    ]),
    'LR' : Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(max_iter=500, C=1.0, solver='lbfgs'))
    ]),
    'SVM (RBF)' : Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    SVC(kernel='rbf', probability=True, C=1.0))
    ]),
    'Random Forest' : Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42))
    ]),
}

print("Modèles configurés:")
for name in MODELS:
    print(f"  - {name}")

In [ ]:
# ── Évaluation avec mesure de latence d'inférence ─────────────────────────────

def evaluate_model(model, X, y_multi, model_name, n_splits=5):
    """
    Évalue un modèle sur un problème multi-label.
    Retourne AUC moyen + temps d'inférence par sample.
    """
    cv = StratifiedKFold(n_splits=n_splits, shuffle=False)
    aucs_per_event = []
    
    # Évaluation par événement (problème multi-label → binarisation)
    for ev_idx, ev_name in enumerate(EVENT_NAMES):
        y_ev = y_multi[:, ev_idx]
        if y_ev.sum() < 10:  # Skip si trop peu d'exemples positifs
            continue
        
        scores = []
        for train_idx, val_idx in cv.split(X, y_ev):
            model.fit(X[train_idx], y_ev[train_idx])
            y_prob = model.predict_proba(X[val_idx])[:, 1]
            scores.append(roc_auc_score(y_ev[val_idx], y_prob))
        aucs_per_event.append(np.mean(scores))
    
    mean_auc = np.mean(aucs_per_event)
    
    # Mesure du temps d'inférence (100 prédictions)
    model.fit(X[:500], y_multi[:500, 0])  # Fit minimal
    sample = X[:1]  # 1 fenêtre
    
    n_bench = 200
    t0 = time.perf_counter()
    for _ in range(n_bench):
        model.predict_proba(sample)
    inference_ms = (time.perf_counter() - t0) / n_bench * 1000
    
    return mean_auc, inference_ms, aucs_per_event


# ── Benchmark complet ──────────────────────────────────────────────────────────
print("Benchmark en cours... (peut prendre quelques minutes)\n")
results = []

for config_name, cfg in WINDOW_CONFIGS.items():
    X_feat, y_feat = features_by_window[config_name]
    window_ms = cfg['window'] / FS * 1000
    
    for model_name, model in MODELS.items():
        print(f"  {model_name} | {cfg['label']}...", end=' ')
        
        try:
            t_start = time.perf_counter()
            auc, infer_ms, auc_per_ev = evaluate_model(model, X_feat, y_feat, model_name)
            train_time = time.perf_counter() - t_start
            
            total_latency = window_ms + infer_ms
            viable = '✅' if total_latency < 300 else ('⚠️' if total_latency < 500 else '❌')
            
            results.append({
                'Modèle'          : model_name,
                'Fenêtre'         : cfg['label'],
                'Window (ms)'     : window_ms,
                'AUC moyen'       : round(auc, 4),
                'Inférence (ms)'  : round(infer_ms, 2),
                'Latence totale (ms)': round(total_latency, 1),
                'Viable RT'       : viable,
            })
            print(f"AUC={auc:.3f} | infer={infer_ms:.1f}ms | latence totale={total_latency:.0f}ms {viable}")
        
        except Exception as e:
            print(f"ERREUR: {e}")

df_results = pd.DataFrame(results)
print("\nBenchmark terminé.")

---
## 6. Benchmark Temps-Réel <a id='6-benchmark'></a>

In [ ]:
# ── Tableau de résultats ───────────────────────────────────────────────────────
print("=" * 85)
print("RÉSULTATS — AUC vs LATENCE vs VIABILITÉ TEMPS-RÉEL")
print("=" * 85)
print(df_results.to_string(index=False))

In [ ]:
# ── Visualisation : trade-off AUC vs Latence ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Trade-off Performance vs Viabilité Temps-Réel', fontweight='bold', fontsize=14)

# Scatter : AUC vs Latence totale
ax = axes[0]
model_names = df_results['Modèle'].unique()
for i, model in enumerate(model_names):
    subset = df_results[df_results['Modèle'] == model]
    ax.scatter(subset['Latence totale (ms)'], subset['AUC moyen'],
               label=model, s=120, color=COLORS[i], zorder=3)
    ax.plot(subset['Latence totale (ms)'], subset['AUC moyen'],
            '--', color=COLORS[i], alpha=0.4)

# Zones de viabilité
ax.axvspan(0, 300, alpha=0.06, color='green', label='Zone viable (<300ms)')
ax.axvspan(300, 500, alpha=0.06, color='orange', label='Zone limite (300-500ms)')
ax.axvspan(500, ax.get_xlim()[1] if ax.get_xlim()[1] > 500 else 5500,
           alpha=0.06, color='red', label='Non viable (>500ms)')
ax.axvline(300, color='green',  ls=':', lw=1.5)
ax.axvline(500, color='orange', ls=':', lw=1.5)

ax.set_xlabel('Latence totale (fenêtre + inférence) en ms', fontsize=11)
ax.set_ylabel('AUC moyen (6 événements)', fontsize=11)
ax.set_title('Performance vs Latence', fontsize=12)
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

# Barplot AUC par fenêtre et modèle
ax2 = axes[1]
pivot = df_results.pivot_table(index='Modèle', columns='Fenêtre', values='AUC moyen')
pivot.plot(kind='bar', ax=ax2, color=[COLORS[0], COLORS[2], COLORS[4]], 
           alpha=0.85, edgecolor='white')
ax2.set_title('AUC par modèle et taille de fenêtre', fontsize=12)
ax2.set_ylabel('AUC moyen')
ax2.set_xlabel('')
ax2.tick_params(axis='x', rotation=20)
ax2.legend(title='Fenêtre', fontsize=8)
ax2.set_ylim([0.5, 1.0])
ax2.axhline(0.9, ls='--', color='gray', alpha=0.5, label='AUC=0.9')

plt.tight_layout()
plt.savefig('benchmark_realtime.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Meilleur modèle viable temps-réel ─────────────────────────────────────────
df_viable = df_results[df_results['Viable RT'] == '✅'].copy()

if not df_viable.empty:
    best = df_viable.loc[df_viable['AUC moyen'].idxmax()]
    print("=" * 55)
    print("MEILLEUR MODÈLE VIABLE EN TEMPS-RÉEL")
    print("=" * 55)
    print(f"Modèle          : {best['Modèle']}")
    print(f"Fenêtre         : {best['Fenêtre']}")
    print(f"AUC moyen       : {best['AUC moyen']:.4f}")
    print(f"Latence totale  : {best['Latence totale (ms)']} ms")
    print(f"Statut          : {best['Viable RT']}")
else:
    print("Aucun modèle viable dans la fenêtre <300ms — voir discussion.")

---
## 7. Discussion & Applicabilité Clinique <a id='7-discussion'></a>

### 7.1 — Synthèse des résultats

| Dimension | Résultat | Interprétation |
|-----------|----------|----------------|
| Meilleur modèle viable (<300ms) | LDA, fenêtre 250ms | AUC 0.813, latence 251ms ✅ |
| Coût de la contrainte temps-réel | Δ AUC ≈ -0.07 vs fenêtre 1s | Dégradation acceptable cliniquement |
| Validation rigoureuse (LOSO) | AUC 0.821 | Confirme les performances sans surestimation |
| Limite principale identifiée | Variabilité inter-séries | Séries 1 et 7 : AUC < 0.72 |

### 7.2 — Contrainte temporelle : le vrai challenge

La fenêtre de labellisation de **±150ms** n'est pas anodine : elle définit la précision temporelle requise par le système.  
En contexte neuroprosthétique :
- Au-delà de **~200-300ms** de latence perçue, le contrôle devient difficile et l'embodiment de la prothèse est compromis
- L'exosquelette de Courtine (NeuroRestore) opère en boucle fermée — chaque prédiction tardive crée une désynchronisation neuromusculaire

Le modèle LDA sur fenêtre 250ms répond à cette contrainte avec une latence totale de **251ms**, soit dans la fenêtre de viabilité clinique.

### 7.3 — Résultat inattendu : le LOSO surpasse le K-Fold

Notre hypothèse initiale était que le K-Fold surestimait les performances via un biais de séquentialité.  
Les résultats infirment cette hypothèse : **AUC LOSO (0.821) > AUC K-Fold (0.788), Δ = -0.034**.

Deux explications probables :
1. **Le K-Fold mélange des contextes hétérogènes** dans train/validation (début vs fin de session, fatigue variable), ce qui dégrade artificiellement les performances mesurées
2. **Le LOSO entraîne sur plus de données cohérentes** (7 séries complètes vs blocs fragmentés), ce qui améliore la qualité du modèle

> Le biais de séquentialité existe, mais son effet net sur ce dataset est faible. La validation LOSO reste méthodologiquement supérieure et c'est la seule défendable en contexte clinique.

### 7.4 — La vraie limite : variabilité inter-séries

L'analyse fold par fold révèle une **hétérogénéité marquée entre sessions** :

| Session | AUC moyen | Statut |
|---------|-----------|--------|
| Série 1 | 0.689 | 🔴 Niveau quasi-aléatoire sur BothReleased (0.495) |
| Série 7 | 0.712 | 🔴 Chute inexpliquée malgré un entraînement sur 7 séries |
| Séries 3,5,6 | 0.873–0.896 | 🟢 Performances stables et solides |

Cette variabilité reflète des phénomènes bien documentés en BCI réel :
- **Dérive du signal** entre sessions (impédance des électrodes, positionnement du casque)
- **Fatigue cognitive et motrice** affectant la qualité du signal M1
- **Apprentissage moteur** : les premières sessions d'un sujet naïf produisent un signal moins stable

> **Implication clinique directe** : la limite de ce système n'est pas l'algorithme ni la latence — c'est la robustesse inter-sessions. C'est précisément pour cette raison que les systèmes BCI cliniques (BrainGate, NeuroRestore) intègrent une **phase de calibration en début de chaque session**.

### 7.5 — Limites générales de ce travail

1. **Dataset public ≠ conditions cliniques** : sujets neurotypiques, environnement contrôlé, pas d'artefacts écologiques
2. **Un seul sujet analysé** : les conclusions sur la variabilité inter-séries demandent une validation sur les 12 sujets
3. **Pipeline offline** : la latence mesurée (251ms) est une estimation — un système embarqué introduit des contraintes supplémentaires
4. **Features génériques** : les matrices de covariance et band power ne sont pas optimisées pour la tâche — CSP ou Riemannian geometry pourraient améliorer les performances

### 7.6 — Perspectives

- **Généralisation inter-sujets** : étendre le LOSO aux 12 sujets pour caractériser la variabilité inter-individuelle
- **Calibration adaptative** : LDA avec mise à jour bayésienne en début de session pour compenser la dérive du signal
- **Géométrie Riemannienne** : approche de référence actuelle en BCI (Alexandre Barachant, vainqueur de cette compétition) — opère directement sur les matrices de covariance
- **EEGNet** : architecture DL légère et spécialisée EEG, potentiellement plus robuste sur les sessions difficiles (Séries 1 et 7)
- **Hardware embarqué** : validation de la latence réelle sur système dédié

---
## 7.5 — Visualisations complémentaires : AUC par événement & Courbes ROC <a id='7-viz'></a>

Tous les événements ne sont pas égaux face au décodage EEG.  
Certains ont une signature neurale plus marquée (ex: HandStart = initiation volontaire = fort potentiel de préparation motrice),  
d'autres sont plus ambigus (ex: BothStartLoadPhase = ajustement fin de force = signal plus subtil).

On analyse ici les performances **par événement** pour identifier les forces et faiblesses du modèle.

In [ ]:
# ── AUC par événement — Meilleur modèle viable (LDA, fenêtre 250ms) ────────────
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve

X_rt, y_rt = features_by_window['short_250ms']
model_best  = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LinearDiscriminantAnalysis())
])

cv = StratifiedKFold(n_splits=5, shuffle=False)
auc_per_event   = []
roc_per_event   = []   # (fpr, tpr) pour chaque événement

for ev_idx, ev_name in enumerate(EVENT_NAMES):
    y_ev     = y_rt[:, ev_idx]
    y_prob_all = np.zeros(len(y_ev))
    
    for train_idx, val_idx in cv.split(X_rt, y_ev):
        model_best.fit(X_rt[train_idx], y_ev[train_idx])
        y_prob_all[val_idx] = model_best.predict_proba(X_rt[val_idx])[:, 1]
    
    auc = roc_auc_score(y_ev, y_prob_all)
    fpr, tpr, _ = roc_curve(y_ev, y_prob_all)
    
    auc_per_event.append(auc)
    roc_per_event.append((fpr, tpr, auc))
    print(f"  {ev_name:<25} AUC = {auc:.4f}")

print(f"\n  Moyenne globale          AUC = {np.mean(auc_per_event):.4f}")

In [ ]:
# ── Visualisation : AUC par événement ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Analyse par événement — LDA | Fenêtre 250ms', fontweight='bold', fontsize=14)

# Barplot AUC par événement
ax1 = axes[0]
bars = ax1.barh(EVENT_NAMES, auc_per_event,
                color=[COLORS[i] for i in range(N_EVENTS)],
                alpha=0.85, edgecolor='white', height=0.6)
ax1.axvline(0.5,  ls='--', color='red',   lw=1.2, alpha=0.7, label='Chance (0.5)')
ax1.axvline(np.mean(auc_per_event), ls='--', color='gray', lw=1.2,
            label=f'Moyenne ({np.mean(auc_per_event):.3f})')
ax1.set_xlim([0.4, 1.0])
ax1.set_xlabel('AUC ROC', fontsize=11)
ax1.set_title('AUC par événement moteur', fontsize=12)
ax1.legend(fontsize=9)

# Annotation des valeurs
for bar, auc in zip(bars, auc_per_event):
    ax1.text(auc + 0.005, bar.get_y() + bar.get_height()/2,
             f'{auc:.3f}', va='center', fontsize=9, fontweight='bold')

# Courbes ROC superposées
ax2 = axes[1]
for i, (fpr, tpr, auc) in enumerate(roc_per_event):
    ax2.plot(fpr, tpr, color=COLORS[i], lw=1.8,
             label=f'{EVENT_NAMES[i]} (AUC={auc:.3f})', alpha=0.85)

ax2.plot([0,1], [0,1], 'k--', lw=1, alpha=0.4, label='Chance')
ax2.fill_between([0,1], [0,1], alpha=0.03, color='gray')
ax2.set_xlabel('Taux de Faux Positifs (FPR)', fontsize=11)
ax2.set_ylabel('Taux de Vrais Positifs (TPR)', fontsize=11)
ax2.set_title('Courbes ROC par événement', fontsize=12)
ax2.legend(fontsize=8, loc='lower right')
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('roc_per_event.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Interprétation physiologique des résultats par événement ──────────────────
print("=" * 60)
print("INTERPRÉTATION PHYSIOLOGIQUE")
print("=" * 60)

physiologie = {
    'HandStart'           : "Initiation volontaire → fort Bereitschaftspotential (BP) ~1s avant",
    'FirstDigitTouch'     : "Contact → feedback somatosensoriel + ajustement moteur fin",
    'BothStartLoadPhase'  : "Montée en force → signal subtil, difficile à discriminer du précédent",
    'LiftOff'             : "Décollage → pic d'activité M1 contralaterale bien documenté",
    'Replace'             : "Repose → désactivation progressive, signal décroissant",
    'BothReleased'        : "Relâchement → rebond bêta (ERS) post-mouvement",
}

sorted_events = sorted(zip(auc_per_event, EVENT_NAMES), reverse=True)
for auc, ev in sorted_events:
    marker = '🟢' if auc >= 0.85 else ('🟡' if auc >= 0.75 else '🔴')
    print(f"\n{marker} {ev:<25} AUC={auc:.3f}")
    print(f"   → {physiologie[ev]}")

---
## 8. Limites méthodologiques & Biais de validation <a id='8-limites'></a>

### 8.1 — Le problème de la séquentialité

Une caractéristique fondamentale de ce dataset, explicitement mentionnée dans sa documentation, est que **les 6 événements moteurs suivent toujours le même ordre** :

```
HandStart → FirstDigitTouch → BothStartLoadPhase → LiftOff → Replace → BothReleased
```

Cette séquence est **déterministe** : dans ce protocole expérimental contrôlé, LiftOff arrive toujours après BothStartLoadPhase, sans exception.

### 8.2 — Pourquoi c'est un problème pour nos modèles

Notre validation actuelle par **K-Fold standard** découpe les données en blocs arbitraires.  
Ce faisant, elle crée une situation où :

- Des fenêtres temporellement **proches** (ex: juste avant et juste après un événement) se retrouvent dans train ET validation
- Le modèle peut apprendre **la position dans la séquence** plutôt que le signal EEG lui-même
- Autrement dit : il apprend *"après BothStartLoadPhase, LiftOff arrive"* — sans décoder réellement l'intention motrice

C'est une forme de **fuite de données temporelle** (temporal data leakage), particulièrement insidieuse car elle n'est pas visible dans le code.

> **Conséquence directe :** Nos AUC actuels (~0.81 pour LDA en 250ms) sont vraisemblablement **surestimés**.  
> Ils reflètent en partie la structure séquentielle du protocole, pas uniquement la qualité du décodage EEG.

### 8.3 — Illustration du biais

| Validation | Ce que le modèle apprend | AUC mesuré |
|------------|--------------------------|------------|
| K-Fold standard | Signal EEG + structure séquentielle | Surestimé |
| Leave-One-Series-Out | Signal EEG uniquement | Honnête |

### 8.4 — Solution : Leave-One-Series-Out (LOSO)

La validation rigoureuse pour ce type de données est le **Leave-One-Series-Out** :

- Chaque sujet dispose de 8 séries d'entraînement
- On entraîne sur 7 séries, on valide sur la 8ème — puis on tourne
- Les séries de train et de validation sont **temporellement disjointes** : aucune fenêtre adjacente ne peut fuiter
- C'est équivalent à tester le modèle sur une **nouvelle session d'enregistrement**

```
Fold 1 : Train [S2,S3,S4,S5,S6,S7,S8] | Val [S1]
Fold 2 : Train [S1,S3,S4,S5,S6,S7,S8] | Val [S2]
...
Fold 8 : Train [S1,S2,S3,S4,S5,S6,S7] | Val [S8]
```

Cette approche simule également la **variabilité intra-sujet entre sessions** — un modèle BCI réel doit être robuste à la dérive du signal entre deux jours d'enregistrement.

> **La section suivante implémente le LOSO et compare les performances corrigées aux performances initiales.**  
> La différence entre les deux AUC quantifiera précisément le biais introduit par notre validation naïve.

---
## 9. Validation rigoureuse — Leave-One-Series-Out (LOSO) <a id='9-loso'></a>

On réentraîne le meilleur modèle viable (LDA, fenêtre 250ms) avec une validation LOSO.  
L'objectif est double :
1. **Corriger le biais** de séquentialité identifié en section 8
2. **Quantifier l'écart** entre performances naïves et performances honnêtes

> ⏱️ Cette cellule est plus longue à exécuter : elle recharge et retraite les données série par série.

In [ ]:
# ── Leave-One-Series-Out — LDA | Fenêtre 250ms ────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import roc_auc_score

WINDOW_SIZE = FS // 4   # 250ms
STEP_SIZE   = FS // 10  # 100ms
SUBJECT_ID  = 1         # On commence sur le sujet 1

print(f"LOSO — Sujet {SUBJECT_ID} | Fenêtre 250ms | LDA")
print(f"{'=' * 55}")

loso_results = []  # AUC par fold et par événement

for val_series in TRAIN_SERIES:
    train_series = [s for s in TRAIN_SERIES if s != val_series]

    # ── Chargement et feature extraction — TRAIN ──────────────────────────────
    X_train_all, y_train_all = [], []
    for s in train_series:
        try:
            X_s, y_s, _ = load_subject_series(SUBJECT_ID, s)
            X_s = preprocess_signal(X_s, strategy='lowpass_lf')
            X_f, y_f = sliding_window_features(X_s, y_s, WINDOW_SIZE, STEP_SIZE)
            X_train_all.append(X_f)
            y_train_all.append(y_f)
        except FileNotFoundError:
            pass
    X_train = np.vstack(X_train_all)
    y_train = np.vstack(y_train_all)

    # ── Chargement et feature extraction — VALIDATION ─────────────────────────
    X_val_s, y_val_s, _ = load_subject_series(SUBJECT_ID, val_series)
    X_val_s = preprocess_signal(X_val_s, strategy='lowpass_lf')
    X_val, y_val = sliding_window_features(X_val_s, y_val_s, WINDOW_SIZE, STEP_SIZE)

    # ── Entraînement et évaluation par événement ───────────────────────────────
    fold_aucs = []
    for ev_idx, ev_name in enumerate(EVENT_NAMES):
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('clf',    LinearDiscriminantAnalysis())
        ])
        model.fit(X_train, y_train[:, ev_idx])
        y_prob = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val[:, ev_idx], y_prob)
        fold_aucs.append(auc)

    mean_auc = np.mean(fold_aucs)
    loso_results.append(fold_aucs)
    print(f"  Fold Val=Série {val_series} | AUC moyen = {mean_auc:.4f} | "
          + " | ".join([f"{ev[:6]}={a:.3f}" for ev, a in zip(EVENT_NAMES, fold_aucs)]))

# ── Résultats agrégés ──────────────────────────────────────────────────────────
loso_array    = np.array(loso_results)           # [n_folds x n_events]
loso_mean_ev  = loso_array.mean(axis=0)          # AUC moyen par événement
loso_global   = loso_mean_ev.mean()              # AUC global

print(f"\n{'=' * 55}")
print(f"RÉSULTATS LOSO — Sujet {SUBJECT_ID}")
print(f"{'=' * 55}")
for ev, auc in zip(EVENT_NAMES, loso_mean_ev):
    print(f"  {ev:<25} AUC = {auc:.4f}")
print(f"\n  Moyenne globale LOSO     AUC = {loso_global:.4f}")
print(f"  Moyenne globale K-Fold   AUC = 0.7875  (rappel)")
print(f"  Δ AUC (biais estimé)         = {0.7875 - loso_global:+.4f}")

In [ ]:
# ── Comparaison visuelle K-Fold vs LOSO ───────────────────────────────────────
# AUC K-Fold par événement (résultats précédents)
auc_kfold = [0.8133, 0.8626, 0.8678, 0.7938, 0.6889, 0.6984]

x       = np.arange(N_EVENTS)
width   = 0.35

fig, ax = plt.subplots(figsize=(13, 6))
bars1 = ax.bar(x - width/2, auc_kfold,    width, label='K-Fold (biaisé)',
               color='steelblue', alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + width/2, loso_mean_ev, width, label='LOSO (rigoureux)',
               color='darkorange', alpha=0.85, edgecolor='white')

# Annotation des deltas
for i, (kf, lo) in enumerate(zip(auc_kfold, loso_mean_ev)):
    delta = lo - kf
    color = 'green' if delta >= 0 else 'red'
    ax.annotate(f'{delta:+.3f}',
                xy=(x[i] + width/2, lo),
                xytext=(0, 6), textcoords='offset points',
                ha='center', fontsize=9, color=color, fontweight='bold')

ax.axhline(0.5, ls='--', color='gray', lw=1, alpha=0.5, label='Chance')
ax.set_xticks(x)
ax.set_xticklabels(EVENT_NAMES, rotation=20, ha='right')
ax.set_ylabel('AUC ROC', fontsize=11)
ax.set_title('K-Fold standard vs Leave-One-Series-Out — LDA | Fenêtre 250ms\n'
             'Le Δ quantifie le biais introduit par la validation naïve',
             fontweight='bold', fontsize=12)
ax.set_ylim([0.4, 1.05])
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Résumé global
ax.text(0.98, 0.05,
        f"K-Fold global : {np.mean(auc_kfold):.3f}\n"
        f"LOSO global   : {loso_global:.3f}\n"
        f"Δ biais       : {np.mean(auc_kfold) - loso_global:+.3f}",
        transform=ax.transAxes, fontsize=10, va='bottom', ha='right',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('loso_vs_kfold.png', dpi=150, bbox_inches='tight')
plt.show()

### Interprétation des résultats LOSO

Le Δ AUC entre K-Fold et LOSO quantifie précisément **le biais introduit par la validation naïve**.  

Deux scénarios possibles :

| Δ AUC | Interprétation |
|-------|----------------|
| Δ faible (<0.02) | Le biais de séquentialité était limité — nos résultats initiaux étaient déjà proches de la réalité |
| Δ modéré (0.02–0.10) | Surestimation notable — le modèle exploitait partiellement la structure séquentielle |
| Δ élevé (>0.10) | Surestimation majeure — les performances K-Fold n'étaient pas représentatives |

Quelle que soit la valeur, **l'AUC LOSO est la seule métrique défendable** dans un contexte de recherche clinique.

---
## 10. Analyse intra-sujet — Généralisation aux 12 sujets <a id='10-intrasujet'></a>

La section précédente (LOSO) a été réalisée sur le Sujet 1 uniquement. On étend maintenant l'analyse à l'ensemble des 12 sujets pour obtenir des résultats représentatifs et quantifier la **variabilité inter-individuelle**.

**Protocole :** Pour chaque sujet, on applique le même LOSO (8 folds) avec le LDA sur fenêtre 250ms.  
**Objectif :** Obtenir un AUC moyen inter-sujets et caractériser la dispersion — c'est la baseline clinique réaliste.

> ⏱️ Cette cellule est longue : 12 sujets × 8 folds × 6 événements. Prévoir 30-60 minutes sur Kaggle.

In [ ]:
# ── LOSO intra-sujet — 12 sujets ──────────────────────────────────────────────
WINDOW_SIZE = FS // 4   # 250ms
STEP_SIZE   = FS // 10  # 100ms

all_subjects_results = {}  # {subject_id: {'mean_auc': float, 'per_event': list, 'per_fold': list}}

print("LOSO intra-sujet — 12 sujets | LDA | Fenêtre 250ms")
print("=" * 65)

for subj in range(1, N_SUBJECTS + 1):
    print(f"\n── Sujet {subj:02d} ──────────────────────────────────")
    
    fold_aucs_subj = []  # AUC moyen par fold
    event_aucs_subj = [[] for _ in range(N_EVENTS)]  # AUC par événement × fold
    
    for val_series in TRAIN_SERIES:
        train_series = [s for s in TRAIN_SERIES if s != val_series]
        
        # Train
        X_train_all, y_train_all = [], []
        for s in train_series:
            try:
                X_s, y_s, _ = load_subject_series(subj, s)
                X_s = preprocess_signal(X_s, strategy='lowpass_lf')
                X_f, y_f = sliding_window_features(X_s, y_s, WINDOW_SIZE, STEP_SIZE)
                X_train_all.append(X_f)
                y_train_all.append(y_f)
            except FileNotFoundError:
                pass
        
        if not X_train_all:
            continue
        
        X_train = np.vstack(X_train_all)
        y_train = np.vstack(y_train_all)
        
        # Validation
        try:
            X_val_s, y_val_s, _ = load_subject_series(subj, val_series)
        except FileNotFoundError:
            continue
        X_val_s = preprocess_signal(X_val_s, strategy='lowpass_lf')
        X_val, y_val = sliding_window_features(X_val_s, y_val_s, WINDOW_SIZE, STEP_SIZE)
        
        # Évaluation par événement
        fold_ev_aucs = []
        for ev_idx in range(N_EVENTS):
            model = Pipeline([
                ('scaler', StandardScaler()),
                ('clf',    LinearDiscriminantAnalysis())
            ])
            model.fit(X_train, y_train[:, ev_idx])
            y_prob = model.predict_proba(X_val)[:, 1]
            auc = roc_auc_score(y_val[:, ev_idx], y_prob)
            fold_ev_aucs.append(auc)
            event_aucs_subj[ev_idx].append(auc)
        
        fold_mean = np.mean(fold_ev_aucs)
        fold_aucs_subj.append(fold_mean)
        print(f"  Fold Val=Série {val_series} | AUC = {fold_mean:.4f}")
    
    subj_mean = np.mean(fold_aucs_subj)
    subj_ev_means = [np.mean(ev) for ev in event_aucs_subj]
    
    all_subjects_results[subj] = {
        'mean_auc'  : subj_mean,
        'per_event' : subj_ev_means,
        'per_fold'  : fold_aucs_subj
    }
    print(f"  → AUC moyen Sujet {subj:02d} : {subj_mean:.4f}")

# ── Résultats agrégés inter-sujets ────────────────────────────────────────────
all_auc_means = [all_subjects_results[s]['mean_auc'] for s in range(1, N_SUBJECTS+1)]
grand_mean    = np.mean(all_auc_means)
grand_std     = np.std(all_auc_means)
grand_min     = np.min(all_auc_means)
grand_max     = np.max(all_auc_means)

print(f"\n{'=' * 65}")
print(f"RÉSULTATS INTRA-SUJET — 12 SUJETS")
print(f"{'=' * 65}")
for subj in range(1, N_SUBJECTS+1):
    auc = all_subjects_results[subj]['mean_auc']
    marker = '🟢' if auc >= 0.80 else ('🟡' if auc >= 0.70 else '🔴')
    print(f"  Sujet {subj:02d} : AUC = {auc:.4f} {marker}")
print(f"\n  Moyenne  : {grand_mean:.4f}")
print(f"  Std      : {grand_std:.4f}")
print(f"  Min      : {grand_min:.4f}")
print(f"  Max      : {grand_max:.4f}")

In [ ]:
# ── Visualisation intra-sujet ──────────────────────────────────────────────────
subjects   = list(range(1, N_SUBJECTS+1))
auc_means  = [all_subjects_results[s]['mean_auc'] for s in subjects]
auc_stds   = [np.std(all_subjects_results[s]['per_fold']) for s in subjects]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Analyse intra-sujet — LOSO | LDA | Fenêtre 250ms', fontweight='bold', fontsize=14)

# Barplot AUC par sujet avec barres d'erreur
ax1 = axes[0]
colors_subj = ['forestgreen' if a >= 0.80 else ('orange' if a >= 0.70 else 'crimson') for a in auc_means]
bars = ax1.bar(subjects, auc_means, yerr=auc_stds, capsize=4,
               color=colors_subj, alpha=0.85, edgecolor='white')
ax1.axhline(grand_mean, ls='--', color='navy', lw=1.5, label=f'Moyenne ({grand_mean:.3f})')
ax1.axhline(0.5, ls=':', color='gray', lw=1, alpha=0.5, label='Chance (0.5)')
ax1.fill_between([0.5, N_SUBJECTS+0.5], grand_mean - grand_std, grand_mean + grand_std,
                 alpha=0.08, color='navy', label=f'±1 std ({grand_std:.3f})')
ax1.set_xlabel('Sujet', fontsize=11)
ax1.set_ylabel('AUC moyen (LOSO)', fontsize=11)
ax1.set_title('AUC par sujet — variabilité inter-individuelle', fontsize=12)
ax1.set_xticks(subjects)
ax1.set_ylim([0.4, 1.05])
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3, axis='y')

# Heatmap AUC par sujet × événement
ax2 = axes[1]
heatmap_data = np.array([all_subjects_results[s]['per_event'] for s in subjects])
im = ax2.imshow(heatmap_data, aspect='auto', cmap='RdYlGn', vmin=0.5, vmax=1.0)
ax2.set_xticks(range(N_EVENTS))
ax2.set_xticklabels(EVENT_NAMES, rotation=30, ha='right', fontsize=9)
ax2.set_yticks(range(N_SUBJECTS))
ax2.set_yticklabels([f'Sujet {s:02d}' for s in subjects], fontsize=9)
ax2.set_title('AUC par sujet × événement', fontsize=12)
plt.colorbar(im, ax=ax2, label='AUC ROC')

# Annotations des valeurs
for i in range(N_SUBJECTS):
    for j in range(N_EVENTS):
        ax2.text(j, i, f'{heatmap_data[i,j]:.2f}', ha='center', va='center',
                 fontsize=7, color='black' if 0.6 < heatmap_data[i,j] < 0.9 else 'white')

plt.tight_layout()
plt.savefig('intrasubject_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Perspective — Vers un modèle généralisable <a id='11-intersujet'></a>

La calibration individuelle (Section 10) est la norme clinique actuelle. Mais elle impose une contrainte forte : chaque nouveau patient nécessite plusieurs sessions d'enregistrement avant de pouvoir utiliser le système.

**La question exploratoire naturelle :** Peut-on entraîner un modèle sur N-1 sujets et le déployer directement sur un nouveau sujet, **sans calibration** ?
C'est le paradigme **Leave-One-Subject-Out (LOSO inter-sujets)**.

### Protocole envisagé

```
Fold 1  : Train [Sujets 2→12] | Val [Sujet 1]
Fold 2  : Train [Sujets 1,3→12] | Val [Sujet 2]
...
Fold 12 : Train [Sujets 1→11] | Val [Sujet 12]
```

### Résultats attendus

Au vu de la forte variabilité inter-individuelle observée en Section 10 (AUC 0.632 à 0.821), on s'attend à une **dégradation significative** des performances sans calibration. Le Δ entre intra et inter-sujets quantifierait précisément l'apport de la calibration individuelle.

### Pourquoi non réalisé

Cette analyse dépasse les ressources de calcul disponibles dans ce contexte : 12 folds × 11 sujets × 8 séries de feature extraction représentent plusieurs heures de calcul CPU. La sélection du modèle (LDA) n'ayant par ailleurs été validée que sur le Sujet 1, une comparaison multi-modèles sur l'ensemble des sujets serait également nécessaire pour une rigueur complète.

### Perspective technique prioritaire

La **géométrie Riemannienne** (pyRiemann) serait particulièrement adaptée à cette problématique : son invariance aux variations d'amplitude inter-individuelles la rend naturellement plus robuste à la généralisation inter-sujets que notre approche LDA actuelle. C'est la direction prioritaire pour les travaux futurs.

```python
# Implémentation envisagée
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace

pipeline_riemannian = Pipeline([
    ('cov', Covariances(estimator='lwf')),
    ('ts',  TangentSpace(metric='riemann')),
    ('clf', LinearDiscriminantAnalysis())
])
```

In [ ]:
# ── Résumé final ───────────────────────────────────────────────────────────────
print("=" * 65)
print("RÉSUMÉ FINAL — EEG NEUROPROSTHETIC CONTROL")
print("=" * 65)
print(f"\nDataset  : 12 sujets | 32 canaux | 500 Hz | 6 événements")
print(f"Modèle   : LDA | Fenêtre 250ms | Latence 251ms ✅")
print(f"\nQ1 — Viabilité temps-réel ?")
print(f"   Fenêtre 250ms : AUC 0.813 (Sujet 1), latence 251ms → viable ✅")
print(f"   Fenêtre 1s    : AUC 0.895 (Sujet 1), latence 1001ms → non viable ❌")
print(f"\nQ2 — Performances généralisées (LOSO intra-sujet, 12 sujets) ?")
print(f"   AUC moyen : {grand_mean:.3f} ± {grand_std:.3f}")
print(f"   Min : {grand_min:.3f} | Max : {grand_max:.3f}")
print(f"   Forte variabilité inter-individuelle — calibration nécessaire")
print(f"\nQ3 — Modèle généralisable sans calibration ?")
print(f"   Non évalué — contrainte de ressources de calcul")
print(f"   Perspective : Leave-One-Subject-Out + géométrie Riemannienne")
print(f"\nLimites documentées :")
print(f"   - Sélection du modèle validée sur Sujet 1 uniquement")
print(f"   - Protocole séquentiel : biais de séquentialité non entièrement corrigeable")
print(f"   - Dataset en conditions contrôlées ≠ conditions cliniques réelles")
print(f"\n{'=' * 65}")